In [1]:
from pathlib import Path
import numpy as np 
import pandas as pd 
import geopandas as gpd
import xarray as xr
import matplotlib.pyplot as plt
import pickle
import altair as alt

### Load the file output by the process_icesat2.ipynb

In [2]:
atl06sr_gdf = pickle.load(open('../../data/icesat2/sierras_gdf_2018_2021_sa_processed_masked.pkl', 'rb'))

In [3]:
sierras = gpd.read_file('../../data/misc/sierras_polygon_gmba.geojson')

In [4]:
# use open_mfdataset to open all years of swe/sca data at once, then open all sd data at once, and merge the two datasets saving the output file

base_dir = '../../swe_reanalysis_pca/data/download'

swe_sca_files = []
sd_files = []

for year in range(2019, 2022):
    year_folder = Path(base_dir) / str(year)
    swe_sca_files.append(f"{base_dir}/{year}/SWE_SCA_POST_{year}.nc")
    sd_files.append(f"{base_dir}/{year}/SD_POST_{year}.nc")

def preprocess_swe(ds):
    ds = ds.rename({'Latitude': 'y', 'Longitude': 'x'})
    ds = ds.rio.set_spatial_dims('y', 'x', inplace=True)
    return ds

swe_sca_ds = xr.open_mfdataset(
    swe_sca_files,
    combine='by_coords',
    preprocess=preprocess_swe
    # chunks={'Latitude': 1575, 'Longitude': 1125, 'time': 366},
    # parallel=True
)


sd_ds = xr.open_mfdataset(
    sd_files,
    combine='by_coords'
    # chunks={'Latitude': 1575, 'Longitude': 1125, 'time': 366},
    # parallel=True
)

swe_sca_ds['SD_Post'] = sd_ds['SD_Post']


# swe_sca_ds = swe_sca_ds.rename({'Latitude': 'y', 'Longitude': 'x'})
# swe_sca_ds = swe_sca_ds.rio.set_spatial_dims('y', 'x', inplace=True)
swe_sca_ds['SWE_Post'] = swe_sca_ds['SWE_Post'].T
swe_sca_ds['SCA_Post'] = swe_sca_ds['SCA_Post'].T
swe_sca_ds = swe_sca_ds.transpose('time', 'y', 'x')
swe_sca_ds.rio.write_crs('EPSG:4326', inplace=True)
#swe_sca_ds['SD_Post'] = swe_sca_ds['SD_Post'].T

<xarray.Dataset> Size: 23GB
Dimensions:      (time: 1096, y: 1575, x: 1125)
Coordinates:
  * y            (y) float32 6kB 34.0 34.01 34.01 34.02 ... 40.99 40.99 41.0
  * x            (x) float32 4kB -122.0 -122.0 -122.0 ... -117.0 -117.0 -117.0
  * time         (time) datetime64[ns] 9kB 2018-10-01 2018-10-02 ... 2021-09-30
    spatial_ref  int64 8B 0
Data variables:
    SWE_Post     (time, y, x) float32 8GB dask.array<chunksize=(61, 263, 188), meta=np.ndarray>
    SCA_Post     (time, y, x) float32 8GB dask.array<chunksize=(61, 263, 188), meta=np.ndarray>
    SD_Post      (time, y, x) float32 8GB dask.array<chunksize=(61, 263, 188), meta=np.ndarray>

In [5]:
# use open_mfdataset to open all years of swe/sca data at once, then open all sd data at once, and merge the two datasets saving the output file

base_dir = '../../swe_reanalysis_pca/data/download'

swe_sca_files = []
sd_files = []

for year in range(1985, 2022):
    year_folder = Path(base_dir) / str(year)
    swe_sca_files.append(f"{base_dir}/{year}/SWE_SCA_POST_{year}.nc")
    sd_files.append(f"{base_dir}/{year}/SD_POST_{year}.nc")

def preprocess_swe(ds):
    ds = ds.rename({'Latitude': 'y', 'Longitude': 'x'})
    ds = ds.rio.set_spatial_dims('y', 'x', inplace=True)
    return ds

swe_sca_ds_allyrs = xr.open_mfdataset(
    swe_sca_files,
    combine='by_coords',
    preprocess=preprocess_swe
    # chunks={'Latitude': 1575, 'Longitude': 1125, 'time': 366},
    # parallel=True
)


sd_ds_allyrs = xr.open_mfdataset(
    sd_files,
    combine='by_coords'
    # chunks={'Latitude': 1575, 'Longitude': 1125, 'time': 366},
    # parallel=True
)

swe_sca_ds_allyrs['SD_Post'] = sd_ds_allyrs['SD_Post']


# swe_sca_ds = swe_sca_ds.rename({'Latitude': 'y', 'Longitude': 'x'})
# swe_sca_ds = swe_sca_ds.rio.set_spatial_dims('y', 'x', inplace=True)
swe_sca_ds_allyrs['SWE_Post'] = swe_sca_ds_allyrs['SWE_Post'].T
swe_sca_ds_allyrs['SCA_Post'] = swe_sca_ds_allyrs['SCA_Post'].T
swe_sca_ds_allyrs = swe_sca_ds_allyrs.transpose('time', 'y', 'x')
swe_sca_ds_allyrs.rio.write_crs('EPSG:4326', inplace=True)
#swe_sca_ds['SD_Post'] = swe_sca_ds['SD_Post'].T

<xarray.Dataset> Size: 287GB
Dimensions:      (time: 13514, y: 1575, x: 1125)
Coordinates:
  * y            (y) float32 6kB 34.0 34.01 34.01 34.02 ... 40.99 40.99 41.0
  * x            (x) float32 4kB -122.0 -122.0 -122.0 ... -117.0 -117.0 -117.0
  * time         (time) datetime64[ns] 108kB 1984-10-01 ... 2021-09-30
    spatial_ref  int64 8B 0
Data variables:
    SWE_Post     (time, y, x) float32 96GB dask.array<chunksize=(61, 263, 188), meta=np.ndarray>
    SCA_Post     (time, y, x) float32 96GB dask.array<chunksize=(61, 263, 188), meta=np.ndarray>
    SD_Post      (time, y, x) float32 96GB dask.array<chunksize=(61, 263, 188), meta=np.ndarray>

### Get timeseries of median/average reanalysis snow depth:

In [6]:
swe_sca_ds = swe_sca_ds.rename({'y': 'lat', 'x': 'lon'})

In [7]:
swe_sca_ds_allyrs = swe_sca_ds_allyrs.rename({'y': 'lat', 'x': 'lon'})

### Plot a scatter of median is2 and reanalysis by date:

In [12]:
atl06sr_gdf['offset_sd'] = atl06sr_gdf['icesat2_3dep_dif'] - atl06sr_gdf['reanalysis_sd']
atl06sr_gdf['n'] = atl06sr_gdf.groupby('acqdate')['acqdate'].transform('count')

### Calculate metrics for all data:

In [13]:
medians_all = atl06sr_gdf[['icesat2_3dep_dif', 'reanalysis_sd','acqdate','n', 'offset_sd', 'dowy']].dropna().groupby('acqdate').median()


### Calculate metrics for fsca > 0.9:

In [64]:
atl06sr_gdf_fsca9 = atl06sr_gdf[atl06sr_gdf['reanalysis_sca']>0.9]
atl06sr_gdf_fsca9['n'] = atl06sr_gdf_fsca9.groupby('acqdate')['acqdate'].transform('count')

medians_fsca9 = atl06sr_gdf_fsca9[['icesat2_3dep_dif', 'reanalysis_sd','acqdate','n', 'offset_sd', 'dowy']].dropna().groupby('acqdate').median()

/opt/homebrew/Caskroom/miniforge/base/envs/analog_mapping/lib/python3.12/site-packages/geopandas/geodataframe.py:1819: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


### Calculate metrics for < 20 degrees:

In [65]:
atl06sr_gdf_fsca9_20slope = atl06sr_gdf_fsca9[atl06sr_gdf_fsca9['slope']<20]
atl06sr_gdf_fsca9_20slope['n'] = atl06sr_gdf_fsca9_20slope.groupby('acqdate')['acqdate'].transform('count')

medians_fsca9_20slope = atl06sr_gdf_fsca9_20slope[['icesat2_3dep_dif', 'reanalysis_sd','acqdate','n', 'offset_sd', 'dowy']].dropna().groupby('acqdate').median()

/opt/homebrew/Caskroom/miniforge/base/envs/analog_mapping/lib/python3.12/site-packages/geopandas/geodataframe.py:1819: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


### Calculate metrics for n > 100:

In [79]:
atl06sr_gdf_20slope_fsca9_n10 = atl06sr_gdf_20slope_fsca9[atl06sr_gdf_20slope_fsca9['n']>100]
atl06sr_gdf_20slope_fsca9_n10['n'] = atl06sr_gdf_20slope_fsca9_n10.groupby('acqdate')['acqdate'].transform('count')

medians_20slope_fsca9_n100 = atl06sr_gdf_20slope_fsca9_n10[['icesat2_3dep_dif', 'reanalysis_sd','acqdate','n', 'offset_sd', 'dowy']].dropna().groupby('acqdate').median()

/opt/homebrew/Caskroom/miniforge/base/envs/analog_mapping/lib/python3.12/site-packages/geopandas/geodataframe.py:1819: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


In [72]:
bias_all = np.mean(medians_all['icesat2_3dep_dif'] - 
                   medians_all['reanalysis_sd'])
bias_20slope = np.mean(medians_20slope['icesat2_3dep_dif'] - 
                   medians_20slope['reanalysis_sd'])
bias_20slope_fsca9 = np.mean(medians_20slope_fsca9['icesat2_3dep_dif'] -
                     medians_20slope_fsca9['reanalysis_sd'])
bias_20slope_fsca9_n100 = np.mean(medians_20slope_fsca9_n100['icesat2_3dep_dif'] -
                        medians_20slope_fsca9_n100['reanalysis_sd'])

cor_all = np.corrcoef(medians_all['icesat2_3dep_dif'].dropna(),
                        medians_all['reanalysis_sd'].dropna())[0,1]
cor_20slope = np.corrcoef(medians_20slope['icesat2_3dep_dif'].dropna(),
                        medians_20slope['reanalysis_sd'].dropna())[0,1]
cor_20slope_fsca9 = np.corrcoef(medians_20slope_fsca9['icesat2_3dep_dif'].dropna(),
                        medians_20slope_fsca9['reanalysis_sd'].dropna())[0,1]
cor_20slope_fsca9_n100 = np.corrcoef(medians_20slope_fsca9_n100['icesat2_3dep_dif'].dropna(),  
                        medians_20slope_fsca9_n100['reanalysis_sd'].dropna())[0,1]


rmse_all = np.sqrt(np.nanmean((medians_all['icesat2_3dep_dif'] -
                                    medians_all['reanalysis_sd'])**2))
rmse_20slope = np.sqrt(np.nanmean((medians_20slope['icesat2_3dep_dif'] -
                                    medians_20slope['reanalysis_sd'])**2))
rmse_20slope_fsca9 = np.sqrt(np.nanmean((medians_20slope_fsca9['icesat2_3dep_dif'] -
                                    medians_20slope_fsca9['reanalysis_sd'])**2))
rmse_20slope_fsca9_n100 = np.sqrt(np.nanmean((medians_20slope_fsca9_n100['icesat2_3dep_dif'] -
                                    medians_20slope_fsca9_n100['reanalysis_sd'])**2))

ubrmse_all = np.sqrt(rmse_all**2 - bias_all**2)
ubrmse_20slope = np.sqrt(rmse_20slope**2 - bias_20slope**2)
ubrmse_20slope_fsca9 = np.sqrt(rmse_20slope_fsca9**2 - bias_20slope_fsca9**2)
ubrmse_20slope_fsca9_n100 = np.sqrt(rmse_20slope_fsca9_n100**2 - bias_20slope_fsca9_n100**2)

In [66]:
bias_all = np.mean(medians_all['icesat2_3dep_dif'] - 
                   medians_all['reanalysis_sd'])
bias_fsca9 = np.mean(medians_fsca9['icesat2_3dep_dif'] - 
                   medians_fsca9['reanalysis_sd'])
bias_fsca9_20slope = np.mean(medians_fsca9_20slope['icesat2_3dep_dif'] -
                     medians_fsca9_20slope['reanalysis_sd'])
bias_fsca9_20slope_n100 = np.mean(medians_20slope_fsca9_n100['icesat2_3dep_dif'] -
                        medians_20slope_fsca9_n100['reanalysis_sd'])

cor_all = np.corrcoef(medians_all['icesat2_3dep_dif'].dropna(),
                        medians_all['reanalysis_sd'].dropna())[0,1]
cor_fsca9 = np.corrcoef(medians_fsca9['icesat2_3dep_dif'].dropna(),
                        medians_fsca9['reanalysis_sd'].dropna())[0,1]
cor_fsca9_20slope = np.corrcoef(medians_fsca9_20slope['icesat2_3dep_dif'].dropna(),
                        medians_fsca9_20slope['reanalysis_sd'].dropna())[0,1]
cor_fsca9_20slope_n100 = np.corrcoef(medians_20slope_fsca9_n100['icesat2_3dep_dif'].dropna(),  
                        medians_20slope_fsca9_n100['reanalysis_sd'].dropna())[0,1]


rmse_all = np.sqrt(np.nanmean((medians_all['icesat2_3dep_dif'] -
                                    medians_all['reanalysis_sd'])**2))
rmse_fsca9 = np.sqrt(np.nanmean((medians_fsca9['icesat2_3dep_dif'] -
                                    medians_fsca9['reanalysis_sd'])**2))
rmse_fsca9_20slope = np.sqrt(np.nanmean((medians_fsca9_20slope['icesat2_3dep_dif'] -
                                    medians_fsca9_20slope['reanalysis_sd'])**2))
rmse_fsca9_20slope_n100 = np.sqrt(np.nanmean((medians_20slope_fsca9_n100['icesat2_3dep_dif'] -
                                    medians_20slope_fsca9_n100['reanalysis_sd'])**2))

ubrmse_all = np.sqrt(rmse_all**2 - bias_all**2)
ubrmse_fsca9 = np.sqrt(rmse_fsca9**2 - bias_fsca9**2)
ubrmse_fsca9_20slope = np.sqrt(rmse_fsca9_20slope**2 - bias_fsca9_20slope**2)
ubrmse_fsca9_20slope_n100 = np.sqrt(rmse_fsca9_20slope_n100**2 - bias_fsca9_20slope_n100**2)

### same metrics as above but median dif instead of mean dif

In [80]:
bias_all = np.median(medians_all['icesat2_3dep_dif'] - 
                   medians_all['reanalysis_sd'])
bias_fsca9 = np.median(medians_fsca9['icesat2_3dep_dif'] - 
                   medians_fsca9['reanalysis_sd'])
bias_fsca9_20slope = np.median(medians_fsca9_20slope['icesat2_3dep_dif'] -
                     medians_fsca9_20slope['reanalysis_sd'])
bias_fsca9_20slope_n100 = np.median(medians_20slope_fsca9_n100['icesat2_3dep_dif'] -
                        medians_20slope_fsca9_n100['reanalysis_sd'])

cor_all = np.corrcoef(medians_all['icesat2_3dep_dif'].dropna(),
                        medians_all['reanalysis_sd'].dropna())[0,1]
cor_fsca9 = np.corrcoef(medians_fsca9['icesat2_3dep_dif'].dropna(),
                        medians_fsca9['reanalysis_sd'].dropna())[0,1]
cor_fsca9_20slope = np.corrcoef(medians_fsca9_20slope['icesat2_3dep_dif'].dropna(),
                        medians_fsca9_20slope['reanalysis_sd'].dropna())[0,1]
cor_fsca9_20slope_n100 = np.corrcoef(medians_20slope_fsca9_n100['icesat2_3dep_dif'].dropna(),  
                        medians_20slope_fsca9_n100['reanalysis_sd'].dropna())[0,1]


rmse_all = np.sqrt(np.nanmean((medians_all['icesat2_3dep_dif'] -
                                    medians_all['reanalysis_sd'])**2))
rmse_fsca9 = np.sqrt(np.nanmean((medians_fsca9['icesat2_3dep_dif'] -
                                    medians_fsca9['reanalysis_sd'])**2))
rmse_fsca9_20slope = np.sqrt(np.nanmean((medians_fsca9_20slope['icesat2_3dep_dif'] -
                                    medians_fsca9_20slope['reanalysis_sd'])**2))
rmse_fsca9_20slope_n100 = np.sqrt(np.nanmean((medians_20slope_fsca9_n100['icesat2_3dep_dif'] -
                                    medians_20slope_fsca9_n100['reanalysis_sd'])**2))

ubrmse_all = np.sqrt(rmse_all**2 - bias_all**2)
ubrmse_fsca9 = np.sqrt(rmse_fsca9**2 - bias_fsca9**2)
ubrmse_fsca9_20slope = np.sqrt(rmse_fsca9_20slope**2 - bias_fsca9_20slope**2)
ubrmse_fsca9_20slope_n100 = np.sqrt(rmse_fsca9_20slope_n100**2 - bias_fsca9_20slope_n100**2)

In [81]:
x_intervals=[1.5]
y_intervals=[0.5, 0.25, 0 ,-0.25]
metrics_subsets = pd.DataFrame({
    'metric': [bias_all, bias_20slope, bias_20slope_fsca9, bias_20slope_fsca9_n100,
               cor_all, cor_20slope, cor_20slope_fsca9, cor_20slope_fsca9_n100,
               rmse_all, rmse_20slope, rmse_20slope_fsca9, rmse_20slope_fsca9_n100,
               ubrmse_all, ubrmse_20slope, ubrmse_20slope_fsca9, ubrmse_20slope_fsca9_n100],
    'metric_label':['bias', 'bias', 'bias', 'bias',
                    'correlation', 'correlation', 'correlation', 'correlation',
                    'rmse', 'rmse', 'rmse', 'rmse',
                    'ubrmse', 'ubrmse', 'ubrmse', 'ubrmse'],
    'labels': ['All data', 'Slope < 20°', 'Slope < 20°, SCA > 0.9', 'Slope < 20°, SCA > 0.9, n>100',
               'All data', 'Slope < 20°', 'Slope < 20°, SCA > 0.9', 'Slope < 20°, SCA > 0.9, n>100',
               'All data', 'Slope < 20°', 'Slope < 20°, SCA > 0.9', 'Slope < 20°, SCA > 0.9, n>100',
               'All data', 'Slope < 20°', 'Slope < 20°, SCA > 0.9', 'Slope < 20°, SCA > 0.9, n>100'],
    'x_pos': [x_intervals[0]]*16,
    'y_pos': [y_intervals[0], y_intervals[0], y_intervals[0],y_intervals[0], y_intervals[1], y_intervals[1], y_intervals[1], y_intervals[1], 
               y_intervals[2], y_intervals[2], y_intervals[2],y_intervals[2], y_intervals[3], y_intervals[3], y_intervals[3], y_intervals[3]]
})

In [82]:
x_intervals=[1.5]
y_intervals=[0.5, 0.25, 0 ,-0.25]
metrics_subsets = pd.DataFrame({
    'metric': [bias_all, bias_fsca9, bias_fsca9_20slope, bias_fsca9_20slope_n100,
               cor_all, cor_fsca9, cor_fsca9_20slope, cor_fsca9_20slope_n100,
               rmse_all, rmse_fsca9, rmse_fsca9_20slope, rmse_fsca9_20slope_n100,
               ubrmse_all, ubrmse_fsca9, ubrmse_fsca9_20slope, ubrmse_fsca9_20slope_n100],
    'metric_label':['bias', 'bias', 'bias', 'bias',
                    'correlation', 'correlation', 'correlation', 'correlation',
                    'rmse', 'rmse', 'rmse', 'rmse',
                    'ubrmse', 'ubrmse', 'ubrmse', 'ubrmse'],
    'labels': ['All data', 'SCA > 0.9', 'SCA > 0.9, Slope < 20°', 'SCA > 0.9, Slope < 20°, n>100',
               'All data', 'SCA > 0.9', 'SCA > 0.9, Slope < 20°', 'SCA > 0.9, Slope < 20°, n>100',
               'All data', 'SCA > 0.9', 'SCA > 0.9, Slope < 20°', 'SCA > 0.9, Slope < 20°, n>100',
               'All data', 'SCA > 0.9', 'SCA > 0.9, Slope < 20°', 'SCA > 0.9, Slope < 20°, n>100'],
    'x_pos': [x_intervals[0]]*16,
    'y_pos': [y_intervals[0], y_intervals[0], y_intervals[0],y_intervals[0], y_intervals[1], y_intervals[1], y_intervals[1], y_intervals[1], 
               y_intervals[2], y_intervals[2], y_intervals[2],y_intervals[2], y_intervals[3], y_intervals[3], y_intervals[3], y_intervals[3]]
})

In [83]:
metrics_subsets['metric_2f'] = metrics_subsets['metric'].apply(lambda x: f"{x:.2f}")

In [85]:
metrics_subsets

,metric,metric_label,labels,x_pos,y_pos,metric_2f
0,-0.009218,bias,All data,1.5,0.50,-0.01
1,-0.204068,bias,SCA > 0.9,1.5,0.50,-0.20
2,-0.113297,bias,"SCA > 0.9, Slope < 20°",1.5,0.50,-0.11
3,-0.112074,bias,"SCA > 0.9, Slope < 20°, n>100",1.5,0.50,-0.11
4,0.753297,correlation,All data,1.5,0.25,0.75
5,0.724993,correlation,SCA > 0.9,1.5,0.25,0.72
6,0.826744,correlation,"SCA > 0.9, Slope < 20°",1.5,0.25,0.83
7,0.929711,correlation,"SCA > 0.9, Slope < 20°, n>100",1.5,0.25,0.93
8,0.222401,rmse,All data,1.5,0.00,0.22
9,0.682214,rmse,SCA > 0.9,1.5,0.00,0.68


In [86]:
x = np.linspace(0, 1.4, 100) # Creates 100 points between x_min and x_max
y = x
line_data = pd.DataFrame({
    'x': [-1, 3],
    'y': [-1, 3]
})

subset_chart1 = alt.layer(alt.Chart(medians_all).mark_circle(size=100, stroke='black', opacity=0.8).encode(
    x=alt.X('icesat2_3dep_dif:Q', title='ICESat-2 Median Snow Depth (m)', scale=alt.Scale(domain=[-1, 3])),
    y=alt.Y('reanalysis_sd:Q', title='Sampled Reanalysis Median Snow Depth (m)', scale=alt.Scale(domain=[-1, 3])),
    color=alt.Color('n:Q', title='Sample Size', scale=alt.Scale(type='log',scheme='viridis'))
).properties(
    title='Median by Aquisition Date All Data'
),
alt.Chart(line_data).mark_line(color='red', strokeDash=[5, 5]).encode(
    x='x:Q',
    y='y:Q')) + \
    alt.Chart(metrics_subsets[metrics_subsets['labels']=='All data']).mark_text(
        align='left',
        baseline='middle',
        dx=5,
        fontSize=15
    ).transform_calculate(
        annotation_text = "datum.metric_label + ': ' + datum.metric_2f"
    ).encode(
        x=alt.X('x_pos:Q'),
        y=alt.Y('y_pos:Q'),
        text='annotation_text:N' #'metric_label:N'+': '+
    ) 

subset_chart2 = alt.layer(alt.Chart(medians_20slope).mark_circle(size=100, stroke='black', opacity=0.8).encode(
    x=alt.X('icesat2_3dep_dif:Q', title='ICESat-2 Median Snow Depth (m)', scale=alt.Scale(domain=[-1, 3])),
    y=alt.Y('reanalysis_sd:Q', title='Sampled Reanalysis Median Snow Depth (m)', scale=alt.Scale(domain=[-1, 3])),
    color=alt.Color('n:Q', title='Sample Size', scale=alt.Scale(type='log',scheme='viridis'))
).properties(
    title='Median by Aquisition Date Slope < 20°'
),
alt.Chart(line_data).mark_line(color='red', strokeDash=[5, 5]).encode(
    x='x:Q',
    y='y:Q')) + \
    alt.Chart(metrics_subsets[metrics_subsets['labels']=='SCA > 0.9']).mark_text(
        align='left',
        baseline='middle',
        dx=5,
        fontSize=15
    ).transform_calculate(
        annotation_text = "datum.metric_label + ': ' + datum.metric_2f"
    ).encode(
        x=alt.X('x_pos:Q'),
        y=alt.Y('y_pos:Q'),
        text='annotation_text:N' #'metric_label:N'+': '+
    ) 

subset_chart3 = alt.layer(alt.Chart(medians_20slope_fsca9).mark_circle(size=100, stroke='black', opacity=0.8).encode(
    x=alt.X('icesat2_3dep_dif:Q', title='ICESat-2 Median Snow Depth (m)', scale=alt.Scale(domain=[-1, 3])),
    y=alt.Y('reanalysis_sd:Q', title='Sampled Reanalysis Median Snow Depth (m)', scale=alt.Scale(domain=[-1, 3])),
    color=alt.Color('n:Q', title='Sample Size', scale=alt.Scale(type='log',scheme='viridis'))
).properties(
    title='Median by Aquisition Date Slope < 20°, SCA > 0.9'
),
alt.Chart(line_data).mark_line(color='red', strokeDash=[5, 5]).encode(
    x='x:Q',
    y='y:Q')) + \
    alt.Chart(metrics_subsets[metrics_subsets['labels']=='SCA > 0.9, Slope < 20°']).mark_text(
        align='left',
        baseline='middle',
        dx=5,
        fontSize=15
    ).transform_calculate(
        annotation_text = "datum.metric_label + ': ' + datum.metric_2f"
    ).encode(
        x=alt.X('x_pos:Q'),
        y=alt.Y('y_pos:Q'),
        text='annotation_text:N' #'metric_label:N'+': '+
    ) 

subset_chart4 = alt.layer(alt.Chart(medians_20slope_fsca9_n100).mark_circle(size=100, stroke='black', opacity=0.8).encode(
    x=alt.X('icesat2_3dep_dif:Q', title='ICESat-2 Median Snow Depth (m)', scale=alt.Scale(domain=[-1, 3])),
    y=alt.Y('reanalysis_sd:Q', title='Sampled Reanalysis Median Snow Depth (m)', scale=alt.Scale(domain=[-1, 3])),
    color=alt.Color('n:Q', title='Sample Size', scale=alt.Scale(type='log',scheme='viridis'))
).properties(
    title='Median by Aquisition Date Slope < 20°, SCA > 0.9, n>100'
),
alt.Chart(line_data).mark_line(color='red', strokeDash=[5, 5]).encode(
    x='x:Q',
    y='y:Q')) + \
    alt.Chart(metrics_subsets[metrics_subsets['labels']=='SCA > 0.9, Slope < 20°, n>100']).mark_text(
        align='left',
        baseline='middle',
        dx=5,
        fontSize=15
    ).transform_calculate(
        annotation_text = "datum.metric_label + ': ' + datum.metric_2f"
    ).encode(
        x=alt.X('x_pos:Q'),
        y=alt.Y('y_pos:Q'),
        text='annotation_text:N' #'metric_label:N'+': '+
    ) 

combined = alt.hconcat(subset_chart1, subset_chart2, subset_chart3, subset_chart4)
combined

alt.HConcatChart(...)

In [70]:
x = np.linspace(0, 1.4, 100) # Creates 100 points between x_min and x_max
y = x
line_data = pd.DataFrame({
    'x': [-1, 3],
    'y': [-1, 3]
})

subset_chart1 = alt.layer(alt.Chart(medians_all).mark_circle(size=100, stroke='black', opacity=0.8).encode(
    x=alt.X('icesat2_3dep_dif:Q', title='ICESat-2 Median Snow Depth (m)', scale=alt.Scale(domain=[-1, 3])),
    y=alt.Y('reanalysis_sd:Q', title='Sampled Reanalysis Median Snow Depth (m)', scale=alt.Scale(domain=[-1, 3])),
    #color=alt.Color('n_is2:Q', title='Sample Size', scale=alt.Scale(type='log',scheme='viridis'))
).properties(
    title='Median by Aquisition Date All Data'
),
alt.Chart(line_data).mark_line(color='red', strokeDash=[5, 5]).encode(
    x='x:Q',
    y='y:Q')) + \
    alt.Chart(metrics_subsets[metrics_subsets['labels']=='All data']).mark_text(
        align='left',
        baseline='middle',
        dx=5,
        fontSize=15
    ).transform_calculate(
        annotation_text = "datum.metric_label + ': ' + datum.metric_2f"
    ).encode(
        x=alt.X('x_pos:Q'),
        y=alt.Y('y_pos:Q'),
        text='annotation_text:N' #'metric_label:N'+': '+
    ) 

subset_chart2 = alt.layer(alt.Chart(medians_fsca9).mark_circle(size=100, stroke='black', opacity=0.8).encode(
    x=alt.X('icesat2_3dep_dif:Q', title='ICESat-2 Median Snow Depth (m)', scale=alt.Scale(domain=[-1, 3])),
    y=alt.Y('reanalysis_sd:Q', title='Sampled Reanalysis Median Snow Depth (m)', scale=alt.Scale(domain=[-1, 3])),
    #color=alt.Color('n_is2:Q', title='Sample Size', scale=alt.Scale(type='log',scheme='viridis'))
).properties(
    title='Median by Aquisition Date SCA > 0.9'
),
alt.Chart(line_data).mark_line(color='red', strokeDash=[5, 5]).encode(
    x='x:Q',
    y='y:Q')) + \
    alt.Chart(metrics_subsets[metrics_subsets['labels']=='SCA > 0.9']).mark_text(
        align='left',
        baseline='middle',
        dx=5,
        fontSize=15
    ).transform_calculate(
        annotation_text = "datum.metric_label + ': ' + datum.metric_2f"
    ).encode(
        x=alt.X('x_pos:Q'),
        y=alt.Y('y_pos:Q'),
        text='annotation_text:N' #'metric_label:N'+': '+
    ) 

subset_chart3 = alt.layer(alt.Chart(medians_fsca9_20slope).mark_circle(size=100, stroke='black', opacity=0.8).encode(
    x=alt.X('icesat2_3dep_dif:Q', title='ICESat-2 Median Snow Depth (m)', scale=alt.Scale(domain=[-1, 3])),
    y=alt.Y('reanalysis_sd:Q', title='Sampled Reanalysis Median Snow Depth (m)', scale=alt.Scale(domain=[-1, 3])),
    #color=alt.Color('n_is2:Q', title='Sample Size', scale=alt.Scale(type='log',scheme='viridis'))
).properties(
    title='Median by Aquisition Date SCA > 0.9, Slope < 20°'
),
alt.Chart(line_data).mark_line(color='red', strokeDash=[5, 5]).encode(
    x='x:Q',
    y='y:Q')) + \
    alt.Chart(metrics_subsets[metrics_subsets['labels']=='SCA > 0.9, Slope < 20°']).mark_text(
        align='left',
        baseline='middle',
        dx=5,
        fontSize=15
    ).transform_calculate(
        annotation_text = "datum.metric_label + ': ' + datum.metric_2f"
    ).encode(
        x=alt.X('x_pos:Q'),
        y=alt.Y('y_pos:Q'),
        text='annotation_text:N' #'metric_label:N'+': '+
    ) 

subset_chart4 = alt.layer(alt.Chart(medians_20slope_fsca9_n100).mark_circle(size=100, stroke='black', opacity=0.8).encode(
    x=alt.X('icesat2_3dep_dif:Q', title='ICESat-2 Median Snow Depth (m)', scale=alt.Scale(domain=[-1, 3])),
    y=alt.Y('reanalysis_sd:Q', title='Sampled Reanalysis Median Snow Depth (m)', scale=alt.Scale(domain=[-1, 3])),
    color=alt.Color('n:Q', title='Sample Size', scale=alt.Scale(type='log',scheme='viridis'))
).properties(
    title='Median by Aquisition Date SCA > 0.9, Slope < 20°, n>100'
),
alt.Chart(line_data).mark_line(color='red', strokeDash=[5, 5]).encode(
    x='x:Q',
    y='y:Q')) + \
    alt.Chart(metrics_subsets[metrics_subsets['labels']=='SCA > 0.9, Slope < 20°, n>100']).mark_text(
        align='left',
        baseline='middle',
        dx=5,
        fontSize=15
    ).transform_calculate(
        annotation_text = "datum.metric_label + ': ' + datum.metric_2f"
    ).encode(
        x=alt.X('x_pos:Q'),
        y=alt.Y('y_pos:Q'),
        text='annotation_text:N' #'metric_label:N'+': '+
    ) 

combined = alt.hconcat(subset_chart1, subset_chart2, subset_chart3, subset_chart4)
combined

alt.HConcatChart(...)